# MANTF 데이터 구축 파이프라인

## 전체 순서
1. 소송 데이터 로드 및 침해소송 필터링
2. 특허번호 정규화 + USPTO 메타데이터 조인
3. 기업명 정규화 + 대표 원고/피고 선택
4. (원고, 피고, 특허) 트리플 생성
5. train/val/test 분할 + negative sampling → CSV
6. 인덱스 파일 생성
7. 기업 소송 그래프 생성 → comp_graph.npy
8. 특허 BERT 임베딩 생성 → titles/abstracts/claims.npy
9. 특허 DeepWalk 임베딩 생성 → deepwalk.npy

In [ ]:
import re
import random
import numpy as np
import pandas as pd
from pathlib import Path

PACER_DIR  = Path('src')
USPTO_DIR  = Path('src')
OUTPUT_DIR = Path('data')

(OUTPUT_DIR / 'samples').mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / 'features').mkdir(parents=True, exist_ok=True)

## Step 1. 소송 데이터 로드 및 침해소송 필터링

In [ ]:
cases   = pd.read_csv(PACER_DIR / 'cases.csv',   low_memory=False)
patents = pd.read_csv(PACER_DIR / 'patents.csv',  low_memory=False)
names   = pd.read_csv(PACER_DIR / 'names.csv',    low_memory=False)

# 침해소송 필터 (case_type_1 = 1.0, Patent Infringement non-DJ)
inf_cases = cases[cases['case_type_1'] == 1.0].copy()
inf_cases['date_filed'] = pd.to_datetime(inf_cases['date_filed'], errors='coerce')
print(f'침해소송 케이스: {len(inf_cases):,}건')

# Patent 타입 필터 (Application 제외)
lit_patents = patents[
    (patents['case_row_id'].isin(inf_cases['case_row_id'])) &
    (patents['patent_doc_type'].str.strip() == 'Patent')
].copy()
lit_patents['patent'] = lit_patents['patent'].astype(str).str.strip()
print(f'분쟁 특허 레코드: {len(lit_patents):,}건')

## Step 2. 특허번호 정규화

In [ ]:
def normalize_patent(s):
    """
    특허번호 정규화
    예: 06457274→6457274, PP008673→PP8673, D0382190→D382190, Re37721→RE37721
    """
    if pd.isna(s):
        return None
    s = str(s).strip().upper()
    m = re.match(r'^([A-Z]*)(\d+)$', s)
    if not m:
        return None
    prefix, digits = m.group(1), m.group(2)
    return prefix + digits.lstrip('0')

lit_patents['patent_norm'] = lit_patents['patent'].apply(normalize_patent)
lit_patents = lit_patents[lit_patents['patent_norm'].notna()]

litigated_norm_set = set(lit_patents['patent_norm'].unique())
print(f'고유 분쟁 특허 수: {len(litigated_norm_set):,}건')

## Step 3. 기업명 정규화

### 왜 필요한가?
- 특허는 `5696571` 같은 고유 번호가 있어서 바로 사용 가능
- 기업명은 자유 텍스트 → 같은 회사가 다르게 표기됨
  - `Apple Inc.` / `Apple, Inc.` / `APPLE INC` → 모두 동일 회사
- 정규화 없이 쓰면 같은 회사가 여러 노드로 쪼개져서 GCN 학습이 망가짐

In [ ]:
# ── 법인 키워드 패턴 (개인명 vs 기업명 구분용) ──────────────────
CORP_KEYWORDS = re.compile(
    r'\b(inc|corp|ltd|llc|co|company|corporation|incorporated|limited|'
    r'lp|llp|plc|gmbh|ag|sa|nv|bv|srl|industries|systems|technologies|'
    r'solutions|enterprises|holdings|group|labs|laboratory|laboratories|'
    r'international|manufacturing|products|services|associates|partners|'
    r'university|institute|foundation|trust|bank|financial)\b',
    re.IGNORECASE
)

def is_company(name: str) -> bool:
    """법인 키워드가 있으면 True (기업), 없으면 False (개인 추정)"""
    if pd.isna(name):
        return False
    return bool(CORP_KEYWORDS.search(str(name)))

# ── 기업명 정규화 규칙 ───────────────────────────────────────────
NOISE_PATTERNS = [
    (r'\bet\s+al\.?$',                  ''),     # "et al" 제거
    (r',?\s*doing\s+business\s+as.*$',  ''),     # "doing business as ..." 제거
    (r',?\s*d/?b/?a.*$',                ''),     # "dba ..." 제거
    (r',?\s*as\s+partners.*$',          ''),     # "as partners" 제거
    (r'\bincorp\b',                     'inc'),  # Incorp → inc
    (r'\bincorporated\b',               'inc'),
    (r'\bcorporation\b',                'corp'),
    (r'\blimited\b',                    'ltd'),
    (r'\bcompany\b',                    'co'),
    (r'[,\.\-\(\)\/\\&\'\"]+',        ' '),    # 구두점 → 공백
    (r'\s{2,}',                         ' '),    # 연속 공백 → 단일 공백
]

def normalize_company(name: str) -> str | None:
    if pd.isna(name) or str(name).strip().lower() == 'doing business as':
        return None
    s = str(name).strip().lower()
    for pattern, repl in NOISE_PATTERNS:
        s = re.sub(pattern, repl, s, flags=re.IGNORECASE)
    s = s.strip()
    return s if s else None

# 정규화 결과 확인
test_names = [
    'Apple Inc.', 'Apple, Inc.', 'APPLE INC',
    'Contour Chair Lounge,Co.,Inc.',
    'Robert M. Spargo, Sr, as partners',
    'doing business as',
    'Wang Laboratories Inc.,',
]
for n in test_names:
    print(f'  {n!r:45s} → {normalize_company(n)!r}')

In [ ]:
# ── Plaintiff / Defendant 필터 및 정규화 ────────────────────────
pla_def = names[
    (names['case_row_id'].isin(inf_cases['case_row_id'])) &
    (names['party_type'].isin(['Plaintiff', 'Defendant']))
].copy()
pla_def['name_norm'] = pla_def['name'].apply(normalize_company)
pla_def = pla_def[pla_def['name_norm'].notna()]

# ── 케이스별 대표 기업 선택 ──────────────────────────────────────
# 법인 키워드 있는 기업 우선 → 없으면 첫 번째
def pick_representative(group):
    corps = group[group['name'].apply(is_company)]
    return corps.iloc[[0]] if len(corps) > 0 else group.iloc[[0]]

plaintiffs = (
    pla_def[pla_def['party_type'] == 'Plaintiff']
    .groupby('case_row_id', group_keys=False)
    .apply(pick_representative)
    [['case_row_id', 'name_norm']]
    .rename(columns={'name_norm': 'plaintiff'})
)

defendants = (
    pla_def[pla_def['party_type'] == 'Defendant']
    .groupby('case_row_id', group_keys=False)
    .apply(pick_representative)
    [['case_row_id', 'name_norm']]
    .rename(columns={'name_norm': 'defendant'})
)

print(f'대표 원고 확보: {len(plaintiffs):,}건')
print(f'대표 피고 확보: {len(defendants):,}건')

## Step 4. (원고, 피고, 특허) 트리플 생성

In [ ]:
# case_row_id 기준으로 조인
triples = (
    lit_patents[['case_row_id', 'patent_norm']]
    .merge(plaintiffs, on='case_row_id', how='inner')
    .merge(defendants, on='case_row_id', how='inner')
    .merge(inf_cases[['case_row_id', 'date_filed']], on='case_row_id', how='inner')
    .rename(columns={'patent_norm': 'patent'})
    [['case_row_id', 'date_filed', 'plaintiff', 'defendant', 'patent']]
    .drop_duplicates(subset=['plaintiff', 'defendant', 'patent'])
    .dropna()
    .reset_index(drop=True)
)

# 원고 = 피고인 케이스 제거
triples = triples[triples['plaintiff'] != triples['defendant']]

print(f'총 트리플 수: {len(triples):,}건')
print(f'고유 기업 수: {pd.concat([triples["plaintiff"], triples["defendant"]]).nunique():,}개')
print(f'고유 특허 수: {triples["patent"].nunique():,}개')
triples.head()

## Step 5. Train / Val / Test 분할 + Negative Sampling → CSV

**시간순 분할** (랜덤 분할 시 미래로 과거 예측하는 leakage 발생)
- Train: 전체의 80% 이전 날짜
- Val: 80~90%
- Test: 90~100%

**Negative sampling 비율**: neg_type 0, 1, 2 동일한 수로 생성
- neg_type 0: 원고를 다른 기업으로 교체
- neg_type 1: 피고를 다른 기업으로 교체
- neg_type 2: 특허를 다른 특허로 교체
- → 학습 시 0,1을 50% drop해서 최종 비율 1:1:2

In [ ]:
# 시간순 정렬 후 분할
triples_sorted = triples.sort_values('date_filed').reset_index(drop=True)
n = len(triples_sorted)
train_end = int(n * 0.8)
val_end   = int(n * 0.9)

train_df = triples_sorted.iloc[:train_end]
val_df   = triples_sorted.iloc[train_end:val_end]
test_df  = triples_sorted.iloc[val_end:]

print(f'Train: {len(train_df):,}건 ({train_df["date_filed"].min()} ~ {train_df["date_filed"].max()})')
print(f'Val:   {len(val_df):,}건 ({val_df["date_filed"].min()} ~ {val_df["date_filed"].max()})')
print(f'Test:  {len(test_df):,}건 ({test_df["date_filed"].min()} ~ {test_df["date_filed"].max()})')

# Negative sampling pool (train 전체 기업/특허)
all_companies = list(set(train_df['plaintiff'].tolist() + train_df['defendant'].tolist()))
all_patents   = list(train_df['patent'].unique())

In [ ]:
def make_samples(df, all_companies, all_patents, seed=42):
    """각 트리플에 대해 neg_type 0, 1, 2 샘플 생성"""
    random.seed(seed)
    rows = []
    pos_set = set(zip(df['plaintiff'], df['defendant'], df['patent']))

    for _, row in df.iterrows():
        pla, def_, pat = row['plaintiff'], row['defendant'], row['patent']

        # neg_type 0: 원고 교체
        for _ in range(10):  # 최대 10번 시도
            neg_pla = random.choice(all_companies)
            if neg_pla != pla and (neg_pla, def_, pat) not in pos_set:
                rows.append({'pos_pla_id': pla, 'pos_def_id': def_, 'pos_pat_id': pat,
                             'neg_id': neg_pla, 'neg_type': 0})
                break

        # neg_type 1: 피고 교체
        for _ in range(10):
            neg_def = random.choice(all_companies)
            if neg_def != def_ and (pla, neg_def, pat) not in pos_set:
                rows.append({'pos_pla_id': pla, 'pos_def_id': def_, 'pos_pat_id': pat,
                             'neg_id': neg_def, 'neg_type': 1})
                break

        # neg_type 2: 특허 교체
        for _ in range(10):
            neg_pat = random.choice(all_patents)
            if neg_pat != pat and (pla, def_, neg_pat) not in pos_set:
                rows.append({'pos_pla_id': pla, 'pos_def_id': def_, 'pos_pat_id': pat,
                             'neg_id': neg_pat, 'neg_type': 2})
                break

    return pd.DataFrame(rows)

train_samples = make_samples(train_df, all_companies, all_patents)
val_samples   = make_samples(val_df,   all_companies, all_patents)
test_samples  = make_samples(test_df,  all_companies, all_patents)

print(f'Train samples: {len(train_samples):,}행 (neg_type 분포: {train_samples["neg_type"].value_counts().to_dict()})')
print(f'Val samples:   {len(val_samples):,}행')
print(f'Test samples:  {len(test_samples):,}행')

train_samples.to_csv(OUTPUT_DIR / 'samples/train.csv', index=False)
val_samples.to_csv(OUTPUT_DIR / 'samples/val.csv',   index=False)
test_samples.to_csv(OUTPUT_DIR / 'samples/test.csv',  index=False)
print('CSV 저장 완료')

## Step 6. 인덱스 파일 생성

In [ ]:
# 모든 split에 등장하는 기업/특허를 포함
all_used_companies = sorted(set(
    triples['plaintiff'].tolist() + triples['defendant'].tolist()
))
all_used_patents = sorted(set(triples['patent'].tolist()))

# company_index.txt: 기업명,1
with open(OUTPUT_DIR / 'samples/company_index.txt', 'w', encoding='utf-8') as f:
    for name in all_used_companies:
        f.write(f'{name},1\n')

# patent_index.txt: 특허번호,1 (정수로 변환 가능한 것만)
with open(OUTPUT_DIR / 'samples/patent_index.txt', 'w', encoding='utf-8') as f:
    for pat in all_used_patents:
        f.write(f'{pat},1\n')

print(f'company_index: {len(all_used_companies):,}개')
print(f'patent_index:  {len(all_used_patents):,}개')

## Step 7. 기업 소송 그래프 생성 → comp_graph.npy

In [ ]:
company2idx = {name: i for i, name in enumerate(all_used_companies)}

# train 소송 기록에서 원고→피고 엣지 구성
# 논문 방식: 소송 비율 threshold 적용 (최소 N회 이상 소송한 쌍만)
from collections import Counter
edge_counter = Counter()
for _, row in train_df.iterrows():
    edge_counter[(company2idx[row['plaintiff']], company2idx[row['defendant']])] += 1

MIN_EDGE_COUNT = 1  # threshold: 필요 시 높여서 노이즈 제거
edges = [[src, dst] for (src, dst), cnt in edge_counter.items() if cnt >= MIN_EDGE_COUNT]

comp_graph = np.array(edges, dtype=np.int32)
np.save(OUTPUT_DIR / 'features/comp_graph.npy', comp_graph)

print(f'엣지 수: {len(edges):,}')
print(f'저장: {OUTPUT_DIR}/features/comp_graph.npy  shape={comp_graph.shape}')

## Step 8. 특허 BERT 임베딩 생성 → features/*.npy

USPTO `g_patent.tsv` (제목) + `g_patent_abstract.tsv` (초록) 사용  
청구항은 현재 src/에 없으므로 초록으로 대체

In [ ]:
# 소송 특허만 필터링해서 제목/초록 로드 (전체 로드 방지)
litigated_set_int = {int(p) for p in all_used_patents if p.isdigit()}

titles_dict, abstracts_dict = {}, {}

print('g_patent.tsv 로딩 중 (제목)...')
for chunk in pd.read_csv(
    USPTO_DIR / 'g_patent.tsv', sep='\t',
    usecols=['patent_id', 'patent_title'], chunksize=500_000
):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin([str(p) for p in litigated_set_int])]
    titles_dict.update(dict(zip(matched['patent_id'], matched['patent_title'])))
print(f'  제목 확보: {len(titles_dict):,}건')

print('g_patent_abstract.tsv 로딩 중 (초록)...')
for chunk in pd.read_csv(
    USPTO_DIR / 'g_patent_abstract.tsv', sep='\t',
    usecols=['patent_id', 'patent_abstract'], chunksize=500_000
):
    chunk['patent_id'] = chunk['patent_id'].astype(str)
    matched = chunk[chunk['patent_id'].isin([str(p) for p in litigated_set_int])]
    abstracts_dict.update(dict(zip(matched['patent_id'], matched['patent_abstract'])))
print(f'  초록 확보: {len(abstracts_dict):,}건')

In [ ]:
from transformers import AutoTokenizer, AutoModel
import torch

# 특허 도메인 특화 BERT 권장
MODEL_NAME = 'anferico/bert-for-patents'  # 또는 'bert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
bert = AutoModel.from_pretrained(MODEL_NAME)
bert.eval()
device = 'cuda' if torch.cuda.is_available() else 'cpu'
bert = bert.to(device)
print(f'BERT 모델 로드 완료 (device={device})')

def encode_texts(texts: list, batch_size: int = 32) -> np.ndarray:
    """텍스트 리스트 → BERT [CLS] 벡터 배열 (N, 768)"""
    all_embs = []
    for i in range(0, len(texts), batch_size):
        batch = [str(t) if t else '' for t in texts[i:i+batch_size]]
        inputs = tokenizer(
            batch, return_tensors='pt',
            truncation=True, max_length=512, padding=True
        )
        inputs = {k: v.to(device) for k, v in inputs.items()}
        with torch.no_grad():
            out = bert(**inputs)
        all_embs.append(out.last_hidden_state[:, 0, :].cpu().numpy())
    return np.vstack(all_embs)

In [ ]:
# 인덱스 순서대로 텍스트 정렬
title_list    = [titles_dict.get(p, '')    for p in all_used_patents]
abstract_list = [abstracts_dict.get(p, '') for p in all_used_patents]
# 청구항 없으므로 초록으로 대체
claim_list    = abstract_list

print(f'인코딩 대상 특허 수: {len(all_used_patents):,}개')
print('제목 인코딩 중...')
title_emb = encode_texts(title_list)
np.save(OUTPUT_DIR / 'features/pat_titles.npy', title_emb)

print('초록 인코딩 중...')
abstract_emb = encode_texts(abstract_list)
np.save(OUTPUT_DIR / 'features/pat_abstracts.npy', abstract_emb)

print('청구항(초록 대체) 인코딩 중...')
claim_emb = encode_texts(claim_list)
np.save(OUTPUT_DIR / 'features/pat_avg_claims.npy', claim_emb)

print(f'저장 완료: shape={title_emb.shape}')

## Step 9. 특허 DeepWalk 임베딩 생성 → features/deepwalk.npy

`g_us_patent_citation.tsv`에서 소송 특허 중심 인용 그래프 구성 후 DeepWalk

In [ ]:
patent2idx = {pat: i for i, pat in enumerate(all_used_patents)}

print('g_us_patent_citation.tsv 로딩 중 (소송 특허 필터)...')
citation_edges = []
for chunk in pd.read_csv(
    USPTO_DIR / 'g_us_patent_citation.tsv', sep='\t',
    usecols=['patent_id', 'citation_patent_id'], chunksize=1_000_000
):
    chunk['patent_id']         = chunk['patent_id'].astype(str)
    chunk['citation_patent_id'] = chunk['citation_patent_id'].astype(str)
    filtered = chunk[
        chunk['patent_id'].isin(patent2idx) |
        chunk['citation_patent_id'].isin(patent2idx)
    ]
    citation_edges.append(filtered)

citation_df = pd.concat(citation_edges, ignore_index=True)
print(f'인용 엣지 수: {len(citation_df):,}건')

In [ ]:
# edgelist 파일로 저장 후 pecanpy로 DeepWalk 실행
edgelist_path = OUTPUT_DIR / 'features/citation_graph.edgelist'
with open(edgelist_path, 'w') as f:
    for _, row in citation_df.iterrows():
        f.write(f"{row['patent_id']} {row['citation_patent_id']}\n")

print(f'Edgelist 저장: {edgelist_path}')
print('\n아래 명령어로 DeepWalk 실행:')
print('  pip install pecanpy')
print(f'  pecanpy --input {edgelist_path} --output data/features/deepwalk.emb \\')
print('           --workers 8 --dimensions 100 --walk-length 80 --num-walks 10 --mode PreComp')

In [ ]:
# DeepWalk 결과(.emb) → npy 변환
# pecanpy 실행 후 아래 셀 실행

EMB_PATH = OUTPUT_DIR / 'features/deepwalk.emb'
EMB_DIM = 100

emb_dict = {}
with open(EMB_PATH, 'r') as f:
    next(f)  # 첫 줄 헤더 스킵 (노드수 차원수)
    for line in f:
        parts = line.strip().split()
        node_id = parts[0]
        vec = list(map(float, parts[1:]))
        emb_dict[node_id] = vec

# 인덱스 순서대로 정렬, 없는 특허는 zero vector
deepwalk_emb = np.array([
    emb_dict.get(pat, [0.0] * EMB_DIM) for pat in all_used_patents
], dtype=np.float32)

np.save(OUTPUT_DIR / 'features/patent_deepwalk_embedding.npy', deepwalk_emb)
print(f'DeepWalk 임베딩 저장 완료: shape={deepwalk_emb.shape}')

## 최종 파일 확인

In [ ]:
import os
expected_files = [
    'samples/company_index.txt',
    'samples/patent_index.txt',
    'samples/train.csv',
    'samples/val.csv',
    'samples/test.csv',
    'features/comp_graph.npy',
    'features/pat_titles.npy',
    'features/pat_abstracts.npy',
    'features/pat_avg_claims.npy',
    'features/patent_deepwalk_embedding.npy',
]

for f in expected_files:
    path = OUTPUT_DIR / f
    if path.exists():
        size = os.path.getsize(path)
        if f.endswith('.npy'):
            arr = np.load(path)
            print(f'✅ {f:50s} shape={arr.shape}')
        else:
            lines = sum(1 for _ in open(path))
            print(f'✅ {f:50s} {lines:,}행')
    else:
        print(f'❌ {f} (없음)')